1. Web Scraping — Teoría Completa
¿Qué es HTTP?
HTTP (HyperText Transfer Protocol) es el protocolo de comunicación que define cómo se transfieren datos entre un cliente y un servidor en la web.
¿Qué es un protocolo?
Un conjunto de reglas que dos partes acuerdan seguir para comunicarse. Como un idioma común.
¿Qué es HTTPS?
Es HTTP con una capa de seguridad (SSL/TLS) que encripta la comunicación. La S es de Secure. Books to Scrape usa HTTP, Open Library usa HTTPS.

¿Qué es una petición HTTP?
Cuando hacés requests.get(url) estás enviando esto al servidor:
GET /catalogue/category/books/mystery_3/index.html HTTP/1.1
Host: books.toscrape.com
User-Agent: Mozilla/5.0

Tiene tres partes:
Método → GET, POST, PUT, DELETE
URL → la ruta del recurso
Headers → información adicional sobre la petición

Métodos HTTP
GET    → pedir datos (leer)
POST   → enviar datos (crear)
PUT    → actualizar datos completos
DELETE → borrar datos
PATCH  → actualizar datos parcialmente
En scraping usamos casi siempre GET porque estamos leyendo páginas.

¿Qué son los Headers?
Son metadatos que van junto con la petición. Los más importantes:
pythonheaders = {
    "User-Agent": "Mozilla/5.0"  # identifica quién hace la petición
    "Accept": "text/html"        # qué tipo de respuesta aceptamos
    "Authorization": "Bearer TOKEN"  # autenticación en APIs
}
¿Por qué usamos User-Agent?
Algunos servidores bloquean peticiones sin User-Agent porque parecen bots. Al especificar uno parecido al de un navegador real, el servidor nos trata como un usuario normal. Wikipedia nos dio 403 sin User-Agent.

¿Qué es una respuesta HTTP?
El servidor responde con:
HTTP/1.1 200 OK
Content-Type: text/html

<!DOCTYPE html>...
Tiene tres partes:

Status code → 200, 404, 500, etc.
Headers → metadatos de la respuesta
Body → el contenido (HTML, JSON, etc.)

En requests accedemos así:
pythonresponse.status_code  # 200
response.headers      # dict con headers
response.text         # body como string
response.json()       # body parseado como dict (si es JSON)

¿Qué es el DOM?
DOM (Document Object Model) es la representación en árbol del HTML. Cada etiqueta es un nodo del árbol:
document
└── html
    └── body
        └── article.product_pod
            ├── div.image_container
            ├── p.star-rating.Four
            └── h3
                └── a[title="Sharp Objects"]
BeautifulSoup nos permite navegar ese árbol.

¿Qué es BeautifulSoup?
Es una librería que parsea HTML y lo convierte en objetos Python navegables. El nombre viene de un poema de Lewis Carroll.
pythonfrom bs4 import BeautifulSoup

soup = BeautifulSoup(html_string, "html.parser")
¿Qué es un parser?
Un programa que analiza texto con una estructura definida y lo convierte en una representación interna. html.parser es el parser nativo de Python. lxml es más rápido porque está escrito en C.

Métodos principales:
pythonsoup.select("css selector") # retorna lista de elementos
soup.select_one("css selector")  # retorna primer elemento o None
elemento.text                    # texto dentro del elemento
elemento["href"]                 # valor de un atributo
elemento["class"]                # lista de clases CSS

¿Qué son los selectores CSS?
Son patrones para encontrar elementos HTML:
"article"              → por etiqueta
".product_pod"         → por clase
"#main"                → por ID
"article.product_pod"  → etiqueta con clase
"h3 a"                 → a dentro de h3
"p.star-rating"        → p con clase star-rating

¿Qué es urljoin?
Es una función del módulo urllib.parse que combina una URL base con una relativa:
pythonfrom urllib.parse import urljoin

urljoin("http://books.toscrape.com/catalogue/category/books/mystery_3/",
        "page-2.html")
# → "http://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html"

urljoin("http://books.toscrape.com/catalogue/category/books/mystery_3/",
        "../../../sharp-objects_997/index.html")
# → resuelve la ruta relativa correctamente

¿Por qué no concatenar con +?
Porque las rutas relativas con ../ no se resuelven correctamente con concatenación simple.

¿Qué es el rate limiting?
Es un límite que impone un servidor sobre cuántas peticiones podés hacer por unidad de tiempo. Si lo superás el servidor responde con 429 Too Many Requests.
¿Cómo lo manejamos?
pythontime.sleep(0.5)  # pausa entre requests
¿Qué es time.sleep()?
Detiene la ejecución del programa por N segundos. Es del módulo time que viene incluido en Python.

¿Qué es try/except?
Es el manejo de errores en Python. Permite capturar excepciones sin que el programa se detenga:
pythontry:
    # código que puede fallar
    response = requests.get(url, timeout=10)
except requests.exceptions.Timeout:
    # se ejecuta si hay timeout
    print("Timeout")
except requests.exceptions.HTTPError:
    # se ejecuta si hay error HTTP
    print("Error HTTP")
except Exception as e:
    # captura cualquier otro error
    print(f"Error: {e}")
¿Qué es un timeout?
Es el tiempo máximo de espera para una respuesta. Si el servidor no responde en ese tiempo, requests lanza una excepción Timeout. Sin timeout, el programa podría quedarse esperando para siempre.

¿Qué es requests?
Es una librería de Python para hacer peticiones HTTP. Es la más popular para este propósito porque tiene una API muy simple comparada con el módulo nativo urllib.
python# Sin requests (urllib nativo - más complejo)
import urllib.request
req = urllib.request.Request(url, headers=headers)
response = urllib.request.urlopen(req)
html = response.read().decode('utf-8')

# Con requests (más simple)
response = requests.get(url, headers=headers)
html = response.text



2. APIs REST — Teoría Completa
¿Qué es una API?
API significa Application Programming Interface. Es una interfaz que permite que dos programas se comuniquen entre sí.
Analogía simple:
En un restaurante vos no entrás a la cocina a buscar la comida. Le decís al mozo lo que querés y él te lo trae. El mozo es la API — es el intermediario entre vos y el sistema.

¿Qué es REST?
REST (Representational State Transfer) es un estilo de arquitectura para diseñar APIs. Una API que sigue estas reglas se llama API REST o RESTful.
Reglas principales de REST:

Usa HTTP como protocolo
Cada recurso tiene una URL única
Usa los métodos HTTP correctamente (GET para leer, POST para crear, etc.)
Las respuestas son stateless — cada petición es independiente, el servidor no recuerda peticiones anteriores


¿Qué es un endpoint?
Es una URL específica de la API que representa un recurso:
https://openlibrary.org/search.json          → buscar libros
https://openlibrary.org/search/authors.json  → buscar autores
https://openlibrary.org/authors/OL1433006A.json → detalle de un autor
Cada endpoint hace una cosa específica.

¿Qué es JSON?
JSON (JavaScript Object Notation) es el formato más usado para intercambiar datos entre APIs. Es texto plano con una estructura de pares clave-valor:
json{
    "name": "Gillian Flynn",
    "birth_date": "1971-02-24",
    "work_count": 41,
    "key": "OL1433006A"
}
¿Por qué JSON y no XML?
JSON es más liviano, más fácil de leer y más fácil de parsear en la mayoría de los lenguajes modernos.
En Python:
python# JSON → dict de Python
data = response.json()        # convierte automáticamente
data["name"]                  # acceder a un campo
data.get("birth_date", None)  # acceder con valor default si no existe

¿Qué es un parámetro de query?
Son datos que se envían en la URL después del ?:
https://openlibrary.org/search.json?title=Sharp+Objects&limit=1

? → inicio de los parámetros
title=Sharp+Objects → parámetro título
& → separador entre parámetros
limit=1 → parámetro límite

Con requests los pasamos como dict y los construye automáticamente:
pythonparams = {"title": "Sharp Objects", "limit": 1}
requests.get(url, params=params)
# construye: url?title=Sharp+Objects&limit=1

¿Qué es la autenticación en APIs?
Algunas APIs requieren identificarse para usarlas. Los métodos más comunes son:
API Key    → una clave que se envía en el header o la URL
Bearer Token → token de autenticación en el header Authorization
OAuth      → sistema más complejo de autorización
Open Library y Wikipedia no requieren autenticación — son APIs públicas. Por eso las elegimos.

¿Qué es el rate limiting en APIs?
Es un límite de requests por unidad de tiempo que impone la API. Si lo superás responde con 429 Too Many Requests.
¿Cómo lo manejamos en nuestro código?
pythonexcept requests.exceptions.HTTPError as e:
    if e.response.status_code == 429:
        time.sleep(60)  # esperar 60 segundos y reintentar

¿Qué es el cache y por qué lo usamos?
El cache es un almacenamiento temporal que guarda resultados de operaciones costosas para reutilizarlos sin repetir el trabajo.
¿Por qué lo necesitamos?
Sin cache, si el mismo autor aparece en 10 libros → 10 llamadas a la API.
Con cache → 1 llamada, el resto se lee de memoria.
Nuestro cache en memoria:
pythoncache_autores = {}  # dict de Python

if nombre_autor in cache_autores:
    return cache_autores[nombre_autor]  # cache hit
else:
    datos = consultar_api(nombre_autor)
    cache_autores[nombre_autor] = datos  # guardar
    return datos
Cache hit → encontró el dato en cache, no llamó a la API.
Cache miss → no estaba en cache, llamó a la API y guardó el resultado.

¿Qué es la normalización de datos?
Es el proceso de limpiar y estandarizar datos para que sean consistentes.
Problema que tuvimos:
Gillian Flynn → "1971-02-24"  (formato fecha ISO)
George Orwell → "25 June 1903" (formato texto)
Solución:
pythonimport re
años = re.findall(r'\d{4}', str(birth_year_raw))
birth_year = int(años[0]) if años else None
# Ambos → 1971 y 1903 como enteros
¿Qué es regex (expresiones regulares)?
Es un lenguaje para buscar patrones en texto. \d{4} significa "exactamente 4 dígitos consecutivos". re.findall() retorna todas las coincidencias como lista.

Open Library vs Wikipedia — ¿Por qué usamos las dos?
CampoOpen LibraryWikipediaexternal_api_id✅ key❌total_known_works✅ work_count❌birth_year⚠️ inconsistente✅ en descriptioncountry❌ no tiene✅ primera palabra
Cada API aporta lo que la otra no tiene. Por eso las combinamos.

¿Qué es response.raise_for_status()?
Es un método de requests que lanza una excepción automáticamente si el status code es >= 400:
pythonresponse = requests.get(url)
response.raise_for_status()  # lanza HTTPError si 404, 500, etc.
Sin esto tendrías que verificar manualmente:

if response.status_code != 200:
    raise Exception(f"Error: {response.status_code}")

Manejo de errores
¿Qué es una excepción?
Es un evento que ocurre durante la ejecución del programa que interrumpe el flujo normal. Si no la capturamos, el programa se detiene.

try/except/finally
pythontry:
    # código que puede fallar
except TipoDeError:
    # qué hacer si falla
finally:
    # se ejecuta SIEMPRE, haya error o no
finally es útil para cerrar conexiones aunque haya un error:
pythontry:
    conn = sqlite3.connect("books.db")
    # operaciones
except Exception as e:
    print(f"Error: {e}")
finally:
    conn.close()  # siempre se cierra

Errores que manejamos en el proyecto
Errores de red:
pythonrequests.exceptions.Timeout      # servidor tardó más de 10 segundos
requests.exceptions.HTTPError    # respuesta con status >= 400
requests.exceptions.ConnectionError  # no hay conexión
requests.exceptions.RequestException # cualquier error de requests
Errores de datos:
pythonjson.JSONDecodeError  # respuesta no es JSON válido
ValueError            # conversión de tipos fallida
KeyError              # clave no existe en el dict

¿Qué es el manejo de NULL?
Cuando la API no devuelve un dato usamos None en Python que se convierte en NULL en SQLite:
pythonbirth_year = doc.get("birth_date", None)  # None si no existe
En SQL:
sqlWHERE country IS NOT NULL  -- filtrar NULLs
WHERE country IS NULL      -- encontrar NULLs

❓ Posibles preguntas del evaluador
¿Qué diferencia hay entre None y NULL?
None es el valor nulo en Python. NULL es el valor nulo en SQL. Cuando insertamos None en SQLite se guarda como NULL automáticamente.
¿Por qué capturamos Exception genérica al final?
Como red de seguridad para errores inesperados. Primero capturamos los errores específicos y al final capturamos cualquier otro error que no hayamos previsto.
¿Qué pasa si no manejamos los errores?
El programa se detiene en el primer error. En nuestro caso si un timeout no estuviera manejado, el pipeline de 1000 libros se detendría en el primer fallo de red.